# FluctlightDB — LongMemEval-S on Google Colab (GPU)

Runs the **official-style session recall@8** benchmark with v2 harness:
- session granularity (one engram per chat session)
- `--dual-key` (user-only index keys)
- `--query-expand` (multi-query merge)
- **GPU embeddings** via `multi-qa-mpnet-base-dot-v1` (no slow CPU embed server)

## Before you run
1. **Runtime → Change runtime type → GPU** (T4 is enough)
2. Run cells **in order** (first run builds Rust native ~5–10 min)
3. Copy the final JSON block back to your FluctlightDB thread

Repo: [voxmastery/FluctlightDB](https://github.com/voxmastery/FluctlightDB)

In [ ]:
# --- Configuration (edit before run) ---
REPO_URL = "https://github.com/voxmastery/FluctlightDB.git"
REPO_BRANCH = "main"

# "full" = all 500 | "preference" = 30 preference questions | "fast" = lexical only (no GPU embed)
BENCH_PROFILE = "full"  # full | preference | fast
LIMIT = 0  # 0 = all items in profile; set e.g. 50 for a quicker smoke test
TOP_K = 8
EMBED_MODEL = "sentence-transformers/multi-qa-mpnet-base-dot-v1"

print("Profile:", BENCH_PROFILE, "limit:", LIMIT or "all")

In [ ]:
import subprocess, sys, os, shutil

def sh(cmd, **kw):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, **kw)
    if r.returncode:
        raise SystemExit(r.returncode)

# GPU check
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — enable GPU runtime!")

sh("pip install -q maturin sentence-transformers datasets huggingface_hub PyYAML filelock")

if not shutil.which("cargo"):
    sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y")
    os.environ["PATH"] = os.environ["HOME"] + "/.cargo/bin:" + os.environ["PATH"]

if not os.path.isdir("FluctlightDB"):
    sh(f"git clone --depth 1 -b {REPO_BRANCH} {REPO_URL} FluctlightDB")
else:
    sh("cd FluctlightDB && git pull --ff-only || true")

os.environ["PATH"] = os.environ["HOME"] + "/.cargo/bin:" + os.environ["PATH"]
sh("cd FluctlightDB/crates/fluctlight-py && maturin develop --release")
sh("pip install -q -e FluctlightDB/sdks/python")

import fluctlightdb_native as _n
from fluctlightdb import connect_index
print("fluctlightdb native OK", _n.__version__ if hasattr(_n, '__version__') else "")

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download

DATA_PATH = Path("/content/longmemeval_s_cleaned.json")
if not DATA_PATH.is_file():
    try:
        p = hf_hub_download(
            repo_id="xiaowu0162/longmemeval-cleaned",
            filename="longmemeval_s_cleaned.json",
            repo_type="dataset",
        )
        DATA_PATH = Path(p)
    except Exception as e:
        print("HuggingFace download failed:", e)
        print("Fallback: download manually and upload to /content/longmemeval_s_cleaned.json")
        raise

import json
data = json.loads(DATA_PATH.read_text())
print("Loaded", len(data), "questions from", DATA_PATH)

In [ ]:
import sys
from typing import Optional

sys.path.insert(0, "FluctlightDB/benchmarks")
import longmemeval_bench as bench

class GpuEmbedCache:
    """In-process GPU embeddings (replaces HTTP embed sidecar)."""

    def __init__(self, model_name: str):
        from sentence_transformers import SentenceTransformer
        device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
        print(f"Loading {model_name} on {device}...")
        self.model = SentenceTransformer(model_name, device=device)
        self.cache: dict[str, list[float]] = {}
        self._requests = 0

    def _key(self, text: str) -> str:
        return (text or "").strip()[:4000]

    def embed_many(self, texts: list[str]) -> list[Optional[list[float]]]:
        out: list[Optional[list[float]]] = [None] * len(texts)
        missing_i, missing_t = [], []
        for i, t in enumerate(texts):
            k = self._key(t)
            if not k:
                continue
            if k in self.cache:
                out[i] = self.cache[k]
            else:
                missing_i.append(i)
                missing_t.append(k)
        if missing_t:
            unique = list(dict.fromkeys(missing_t))
            vecs = self.model.encode(
                unique, normalize_embeddings=True, batch_size=64, show_progress_bar=True
            )
            for t, v in zip(unique, vecs):
                self.cache[t] = v.tolist()
                self._requests += 1
            for i, t in zip(missing_i, missing_t):
                out[i] = self.cache.get(t)
        return out

    def embed_one(self, text: str) -> Optional[list[float]]:
        return self.embed_many([text])[0]

    def embed(self, text: str) -> Optional[list[float]]:
        return self.embed_one(text)

# Monkey-patch bench to use GPU embedder
bench.EmbedCache = GpuEmbedCache  # type: ignore
print("GpuEmbedCache ready")

In [ ]:
import time
from collections import defaultdict

items = data
if BENCH_PROFILE == "preference":
    items = [x for x in items if x.get("question_type") == "single-session-preference"]
if LIMIT > 0:
    items = items[:LIMIT]

use_fast = BENCH_PROFILE == "fast"
dual_key = True
query_expand = True

class NoEmbed:
    def __init__(self):
        self.cache = {}
        self._requests = 0
    def embed_many(self, texts):
        return [None] * len(texts)
    def embed(self, text):
        return None

embedder = NoEmbed() if use_fast else GpuEmbedCache(EMBED_MODEL)

results = []
hits = 0
t0 = time.perf_counter()

for i, item in enumerate(items):
    row = bench.eval_one(
        item,
        mode="index",
        top_k=TOP_K,
        embedder=embedder,
        fast=use_fast,
        granularity="session",
        metric="session",
        query_expand=query_expand,
        dual_key=dual_key,
    )
    results.append(row)
    if row.get("hit"):
        hits += 1
    if (i + 1) % 5 == 0 or (i + 1) == len(items):
        print(f"[{i+1}/{len(items)}] session_recall@{TOP_K}={hits/(i+1):.1%} ({hits}/{i+1}) last_sec={row.get('sec')}")

wall = time.perf_counter() - t0
by_type = defaultdict(list)
for r in results:
    by_type[str(r.get("question_type") or "unknown")].append(bool(r.get("hit")))

report = {
    "benchmark": "longmemeval_s",
    "platform": "google_colab_gpu",
    "embed_model": EMBED_MODEL if not use_fast else None,
    "profile": BENCH_PROFILE,
    "granularity": "session",
    "metric": "session",
    "query_expand": query_expand,
    "dual_key": dual_key,
    "top_k": TOP_K,
    "questions": len(results),
    "session_recall_at_k": round(hits / len(results), 4) if results else 0.0,
    "hits": f"{hits}/{len(results)}",
    "wall_s": round(wall, 1),
    "sec_per_question": round(wall / max(1, len(results)), 2),
    "embed_cache_size": len(getattr(embedder, "cache", {})),
    "by_type": {k: round(sum(v) / len(v), 4) for k, v in sorted(by_type.items())},
}

out = {"summary": report, "results": results}
print("\n" + "=" * 60)
print("COPY EVERYTHING BELOW THIS LINE")
print("=" * 60)
print(json.dumps(out, indent=2))
print("=" * 60)

out_path = Path("/content/longmemeval_colab_result.json")
out_path.write_text(json.dumps(out, indent=2))
print("Saved:", out_path)
try:
    from google.colab import files
    files.download(str(out_path))
except ImportError:
    pass

## Paste results back

Copy the JSON between the `====` lines and paste it in your FluctlightDB chat.

### Profiles
| `BENCH_PROFILE` | What it runs |
|-----------------|--------------|
| `full` | 500 questions, GPU mpnet + dual-key + query-expand |
| `preference` | 30 preference questions only |
| `fast` | 500 questions, lexical only (no embed, ~fastest) |

Set `LIMIT = 50` in cell 1 for a quick smoke test before the full 500.